# Attempting to Model NGC6569 with PyfalcON


In [ ]:
import numpy as np

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.animation import FuncAnimation

import matplotlib as mpl
mpl.rcParams['animation.embed_limit'] = 200  # Set to 100MB or whatever you need

from IPython.display import HTML

import pandas as pd

import pyfalcon

In [ ]:
import astropy.coordinates as coord
import astropy.units as u
from astropy.constants import G
import gala.coordinates as gc
import gala.dynamics as gd
import gala.potential as gp
from gala.units import galactic
from gala.dynamics import mockstream as ms

import agama

import importlib
import sys
from pathlib import Path

from time import time

START_DIR = Path.cwd().resolve()
NGC6569_DIR = next(
    (path for path in (START_DIR, START_DIR / "ngc6569", START_DIR / "joe's_Code" / "ngc6569")
     if (path / "milkyway").is_dir()),
    None,
)
if NGC6569_DIR is None:
    raise FileNotFoundError("Could not locate joe's_Code/ngc6569/milkyway from the current working directory")

DATA_DIR = NGC6569_DIR / "data"
OUTPUT_DIR = NGC6569_DIR / "output"
MILKYWAY_DIR = NGC6569_DIR / "milkyway"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if str(NGC6569_DIR) not in sys.path:
    sys.path.insert(0, str(NGC6569_DIR))

from leap_frog import kdk_leapfrog_TD
from leap_frog import kdk_leapfrog

#from leap_frog_hdf5 import kdk_leapfrog_hdf5, load_simulation_hdf5

In [ ]:
# default Astropy Galactocentric frame parameters to the values adopted in Astropy v4.0:
_ = coord.galactocentric_frame_defaults.set('v4.0')

# set Agama units 
# working units: 1 Msun, 1 kpc, 1 km/s
agama.setUnits(length=1*u.kpc, velocity=1*u.km/u.s, mass=1*u.Msun)
print("Newton G in Agama units,",agama.G)

# Check the current unit system
print("Current Agama units:")
print(f"Length unit: {agama.getUnits()['length']}")
print(f"Velocity unit: {agama.getUnits()['velocity']}")  
print(f"Time unit: {agama.getUnits()['time']}")
print(f"Mass unit: {agama.getUnits()['mass']}")

agama_time_unit = agama.getUnits()["time"]
print(agama_time_unit)

In [ ]:
# Use the Hunter rotating potential 
pot_ext = agama.Potential(str(MILKYWAY_DIR / "MWPotentialHunter24_full.ini")) 
pot_rot = agama.Potential(str(MILKYWAY_DIR / "MWPotentialHunter24_rotating.ini")) 
pot_bovy = agama.Potential(str(MILKYWAY_DIR / "MWPotential2014.ini")) 

pot_use = pot_rot

In [ ]:
pot_ext

In [ ]:
# NG6569 coordinates 

c = coord.SkyCoord(ra = 273.412*u.degree, dec = -31.827*u.degree,
                        distance=(10.5)*u.kpc,
                        pm_ra_cosdec= -4.125*u.mas/u.yr,
                        pm_dec= -7.354*u.mas/u.yr,
                        radial_velocity= -49.82*u.km/u.s)

# transform to galactic centeric 
c_gc = c.transform_to(coord.Galactocentric).data
print(c_gc._differentials)

In [ ]:
# creat phase space object
w0 = gd.PhaseSpacePosition(c_gc)
print("initial position :", w0.pos)
print("initial velocity :", w0.vel)
print("Need to convert velocity to km/s")

pos_0 = np.r_[w0.pos.x.value, w0.pos.y.value, w0.pos.z.value]
vel_0 = np.r_[w0.vel.d_x.to(u.km/u.s).value, 
              w0.vel.d_y.to(u.km/u.s).value,
              w0.vel.d_z.to(u.km/u.s).value]
print("position", pos_0) 
print("velocity", vel_0) 

In [ ]:
# check against 
# -31.81767578935911 km / s -174.361128405216 km / s 23.931083813686854 km / s

In [ ]:
# # check the enclosed mass at rmax of the NFW profile, so that it seems reasonable 
# rmax = alpha*r_scale
# pot["halo"].mass_enclosed(np.array([rmax.value, 0, 0]))

In [ ]:
# integrate orbit 
tfin= -200*u.Myr
nt=2000
t_eval = np.linspace(0, tfin, nt)
t_scipy = t_eval.to(u.Gyr).value/agama_time_unit.to(u.Gyr).value
t_scipy

In [ ]:
def rhs(t,state): 
    
    pos = state[:3]
    vel = state[3:]

    acc = pot_use.force(pos, t=t)
    return np.r_[vel, acc] 

# integrate with solve_ivp
from scipy.integrate import solve_ivp

state_0 = np.r_[pos_0, vel_0] 
print(state_0, state_0.shape)

t_span = (0, t_scipy[-1]) 
sol = solve_ivp(rhs, t_span, state_0, t_eval =t_scipy, rtol=1e-10, atol=1e-10)
orbit = sol["y"]

In [ ]:
x = orbit[0,:]
y = orbit[1,:]
z = orbit[2,:]
vx = orbit[3,:]
vy = orbit[4,:]
vz = orbit[5,:]

In [ ]:
# Agama orbit
# Calculate orbit
orbit = agama.orbit(potential=pot_use, 
                   ic=state_0, 
                   time=t_span[1],      # Total integration time
                   trajsize=nt)   # Number of output points

In [ ]:
orbit[1].shape

In [ ]:
# 2-D orbit figures 
fig = plt.figure(figsize=(14, 4))

ax = fig.add_subplot(131)
ax.set_xlabel("x [kpc]", size=20) 
ax.set_ylabel("y [kpc]", size=20)
ax.plot(x,y, label="from scratch")
ax.plot(orbit[1][:,0], orbit[1][:,1], "--", label="Agama") 
ax.legend()

ax = fig.add_subplot(132)
ax.set_xlabel("x [kpc]", size=20) 
ax.set_ylabel("z [kpc]", size=20)
ax.plot(x,z)
ax.plot(orbit[1][:,0], orbit[1][:,2], "--") 

ax = fig.add_subplot(133)
ax.set_xlabel("y [kpc]", size=20) 
ax.set_ylabel("z [kpc]", size=20)
ax.plot(y,z)
ax.plot(orbit[1][:,1], orbit[1][:,2], "--") 
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "ngc6569_oribit.pdf")

## GyrfalcON manuals
* https://teuben.github.io/nemo/man_html/gyrfalcON.1.html
* 



In [ ]:
# Create figure
fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111, projection='3d')
    
# Create scatter plot with color mapping by radius
ax.plot(x,y,z, alpha = .5) 
ax.set_xlabel('X (pc)')
ax.set_ylabel('Y (pc)')
ax.set_zlabel('Z (pc)')
fig.savefig(OUTPUT_DIR / "ngc6569_3d_orbit.pdf") 


In [ ]:
# Define the parameters for your King model
W0_value = 7.0  # Example W0 value
# create an isolated star cluster
r_scale = 1/1000
m = 2.3*1e5*(2)
pot_sat = agama.Potential(type='king', W0=W0_value, scaleRadius=r_scale, mass=m)
df_sat = agama.DistributionFunction(type='quasispherical', potential=pot_sat)
Nbody = 150000
xv, mass = agama.GalaxyModel(pot_sat, df_sat).sample(Nbody)

r_agama = np.sqrt(xv[:,0]**2 + xv[:,1]**2 + xv[:,2]**2) 
v_agama = np.sqrt(xv[:,3]**2 + xv[:,4]**2 + xv[:,5]**2) 

print("Agama G:", agama.G)

cluster_data = np.c_[mass, xv]
cluster_data.shape
np.savetxt(DATA_DIR / "cluster_data.txt", cluster_data)

In [ ]:

# look at positions in 3D
fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111, projection='3d')
    
# Create scatter plot
ax.scatter(xv[:,0], xv[:,1], xv[:,2], alpha=0.1, s=20)
    
# Labels and title
ax.set_xlabel('X [kpc]', fontsize=12)
ax.set_ylabel('Y [kpc]', fontsize=12)
ax.set_zlabel('Z [kpc]', fontsize=12)
    

In [ ]:
fig = plt.figure(figsize=(14,6))

ax=fig.add_subplot(121)
ax.set_xlabel("radial distance", size=20) 
ax.hist(r_agama, bins=30, edgecolor="black", alpha=.5, label ="Agama")

ax.axvline(r_scale, color="black", label="core radius")
ax.legend(loc=0) 

ax=fig.add_subplot(122)
ax.set_xlabel("speed", size=20) 
ax.hist(v_agama, bins=30, edgecolor="black", alpha=.5, label ="Agama")

ax.legend(loc=0) 


In [ ]:
pos_0 = np.r_[x[-1], y[-1], z[-1]]
vel_0 = np.r_[vx[-1], vy[-1], z[-1]]
print(pos_0)
print(vel_0) 

In [ ]:
print("make this a function")

def shift_to_gc(xv, pos_0, vel_0): 

    x = xv[:,0] + pos_0[0]
    y = xv[:,1] + pos_0[1]
    z = xv[:,2] + pos_0[2]

    vx = xv[:,3] + vel_0[0]
    vy = xv[:,4] + vel_0[1]
    vz = xv[:,5] + vel_0[2]


    print("checks")
    print(np.mean(x), np.mean(y), np.mean(z))
    print(pos_0)


    print(np.mean(vx), np.mean(vy), np.mean(vz))
    print(vel_0)

    return np.column_stack((x, y, z)), np.column_stack((vx, vy, vz))


In [ ]:
# pos_0 = np.column_stack((x, y, z))
# vel_0 = np.column_stack((vx, vy, vz))
# vel_0.shape

pos_0 , vel_0 = shift_to_gc(xv, pos_0, vel_0)

In [ ]:
np.sum(mass/1e5)
np.savetxt(DATA_DIR / "ngc6569_mass.txt", mass)

In [ ]:
time_unit=u.kpc.to(u.km)*u.s.to(u.Gyr)
time_unit

In [ ]:
tmax = -tfin.to(u.Gyr).value/time_unit
print("maximum time", tmax)
print(t_scipy[-1])

In [ ]:
kmax=16
tau = 2**(-kmax)*time_unit
print("time step:", tau) 
nt=int(tmax/tau) + 1
print("number of time steps:", nt)

eps = 1/1000  # II 
eps = .1/1000 # I 
eps_power = -4
eps = (2**eps_power)/1000
print("softening length:", eps) 

In [ ]:
(nt)*tau

In [ ]:
print(nt)

In [ ]:
downsample=20
#filename="ngc_6569_runI"
#nt=1000

In [ ]:
pot_use

In [ ]:
t1 = time()
sim_data = kdk_leapfrog_TD(pot_use, pos_0, vel_0, 
                        mass, nt, tau, agama.G, eps, 
                        time_unit, downsample,last_snapshot=False)
t2=time()
print("run time", t2-t1, (t2-t1)/60)

In [ ]:
len(sim_data)

In [ ]:
#gc_data = load_simulation_hdf5(sim_data)

In [ ]:
#gc_data["metadata"]

In [ ]:
# get time array 
time_array=[]
for i in range(len(sim_data)): 

    t = sim_data[i]["time"]
    time_array.append(t*1000) 

time_array=np.array(time_array)    

In [ ]:
from bound_funcs import get_bound_particles

bound_data = get_bound_particles(sim_data, mass, pot_use, agama.G, time_unit)

In [ ]:
# Create figure
fig = plt.figure(figsize=(5, 5))
ax = fig.add_subplot(111)  # Fixed: added subplot number
ax.set_aspect('equal')     # Fixed: proper way to set equal aspect
ax.set_xlabel('X (kpc)')
ax.set_ylabel('Y (kpc)')
L = 4
ax.set_xlim(-L, L)
ax.set_ylim(-L, L)

ax.plot(orbit[1][:,0],orbit[1][:,1], color="black", alpha=.1, label="Agama")
# gala orbit 
#ax.plot(orbit.pos.x,orbit.pos.y, color="black", alpha=.5, label="Gala")
pts, = ax.plot([], [], "o", color="blue", alpha=0.25, markersize=1)  # Added markersize for visibility
plt.close()

# Initialization function
def init():
    pts.set_data([], []) 
    return pts, 

def draw(i):  # Fixed: use 'i' instead of undefined 'nt'

    data = sim_data[i]
    pos = data["pos"]


    x = pos[:,0]
    y = pos[:,1]
    z = pos[:,2] 
    
    pts.set_data(x, y)  # Fixed: missing closing bracket and proper syntax

    time = np.round(time_array[i] + tfin.value,0) 
    
    ax.set_title(f'NGC6569 5 (X-Y Plane) - Time: {time} Myr')  # Fixed: use 'i' instead of 'nt'
    return pts, 
    
anim = FuncAnimation(fig, draw, init_func=init, frames=len(sim_data), interval=50, blit=True)
anim.save(str(OUTPUT_DIR / "6569_2d_II.mp4")) 
HTML(anim.to_jshtml())

In [ ]:
len(sim_data)

In [ ]:
def trajectories(bound_data, sim_data): 
    """
    Extract time evolution trajectories of bound cluster properties from simulation data.
    
    This function processes the output from bound particle tracking to create clean
    time series arrays for analysis and plotting of cluster evolution.
    
    Parameters:
    -----------
    bound_data : list of dict
        List of dictionaries containing bound particle data at each timestep.
        Each dictionary should contain keys: 'total mass', 'total energy', 'pos', 'vel'
    sim_data : list of dict
        List of dictionaries containing simulation data at each timestep.
        Each dictionary should contain key: 'time'
        
    Returns:
    --------
    dict : Dictionary containing trajectory arrays
        'mass' : array, shape (n,) - Total mass of bound particles vs time
        'energy' : array, shape (n,) - Total energy of bound particles vs time  
        'time' : array, shape (n,) - Time array
        'pos' : array, shape (n, 3) - Center of mass position vs time
        'vel' : array, shape (n, 3) - Center of mass velocity vs time

    """
    
    n = len(bound_data)
    
    # Validate input lengths match
    if len(sim_data) != n:
        raise ValueError(f"Length mismatch: bound_data has {n} entries, sim_data has {len(sim_data)}")
    
    # Initialize trajectory arrays
    time_array = np.zeros(n)
    mass_traj = np.zeros(n)
    energy_traj = np.zeros(n)
    r_traj = np.zeros((n, 3))
    d_traj = np.zeros(n)
    v_traj = np.zeros((n, 3))
    
    # Extract data for each timestep
    for i in range(n): 
        # Extract bound particle properties
        mass_traj[i] = bound_data[i]["total mass"]
        energy_traj[i] = bound_data[i]["total energy"]
        r_traj[i, :] = bound_data[i]["pos"]
        v_traj[i, :] = bound_data[i]["vel"]
        d_traj[i] = np.linalg.norm(r_traj[i,:]) 
        
        # Extract time from simulation data
        time_array[i] = sim_data[i]["time"] 
    
    # Package results into dictionary
    traj = {
        'mass': mass_traj,
        'energy': energy_traj,
        'time': time_array,
        'pos': r_traj,
        'vel': v_traj,
        'distance': d_traj
    }
    
    return traj

traj = trajectories(bound_data, sim_data)

In [ ]:
# mass_traj = []
# traj =[] 
# v_traj =[]

# for i in range(len(bound_data)): 

#     m = bound_data[i]["total mass"]
#     pos = bound_data[i]["pos"]
#     vel = bound_data[i]["vel"]
#     mass_traj.append(m) 
#     traj.append(pos) 
#     v_traj.append(vel) 

# mass_traj=np.array(mass_traj)/np.sum(mass)
# traj = np.array(traj)
# v_traj = np.array(v_traj)


# fig = plt.figure()
# fig.suptitle('Mass Loss', size=30)
# ax = fig.add_subplot()
# ax.set_ylabel(r"$M/M_0$", size=20) 
# ax.set_xlabel("Myr", size=20) 
# ax.plot(time_array + tfin.value, mass_traj)

In [ ]:
def cluster_frame(r_cm, v_cm, pos, vel): 
    """
    Compute unit vectors for cluster frame and transform positions to that frame.
    
    Parameters:
    -----------
    r_cm : array, shape (3,)
        Position vector from galactic center to cluster center of mass
    v_cm : array, shape (3,)
        Velocity vector of cluster center of mass  
    pos : array, shape (N, 3)
        Positions of N particles
    vel : array, shape (N, 3)
        Velocities of N particles
        
    Returns:
    --------
    tuple : (x_hat, y_hat, z_hat, pos_cluster_frame, vel_cluster_frame, omega_vec)
        x_hat, y_hat, z_hat : unit vectors in cluster frame
        pos_cluster_frame : array, shape (N, 3) - positions in cluster frame
        vel_cluster_frame : array, shape (N, 3) - velocities in cluster frame
        omega_vec : array, shape (3,) - angular velocity vector of cluster
    """
    
    # x-unit vector points toward galactic center 
    x_hat = -r_cm / np.linalg.norm(r_cm)
    
    # z-unit vector is perpendicular to orbital plane (r × v direction)
    L_vec = np.cross(r_cm, v_cm)
    z_hat = L_vec / np.linalg.norm(L_vec)
    
    # y-unit vector completes right-handed system (z × x)
    y_hat = np.cross(z_hat, x_hat)
    
    # Angular velocity vector: ω = L / |r|²
    r_cm_mag_sq = np.dot(r_cm, r_cm)
    omega_vec = L_vec / r_cm_mag_sq
    
    # Transform positions to cluster frame
    # Each row of pos gets dotted with each unit vector
    x_coords = np.dot(pos, x_hat)  # shape (N,)
    y_coords = np.dot(pos, y_hat)  # shape (N,)
    z_coords = np.dot(pos, z_hat)  # shape (N,)
    
    # Stack coordinates to form (N, 3) array
    pos_cluster_frame = np.column_stack([x_coords, y_coords, z_coords])
    
    # Transform velocities to cluster frame
    # Each row of vel gets dotted with each unit vector
    vx_coords = np.dot(vel, x_hat)  # shape (N,)
    vy_coords = np.dot(vel, y_hat)  # shape (N,)
    vz_coords = np.dot(vel, z_hat)  # shape (N,)
    
    # Stack velocity components to form (N, 3) array
    vel_cluster_frame = np.column_stack([vx_coords, vy_coords, vz_coords])
    
    return x_hat, y_hat, z_hat, pos_cluster_frame, vel_cluster_frame, omega_vec
        
  

In [ ]:
def get_cluster_data(bound_data, sim_data): 
    n=len(bound_data) 

    cluster_data = []

    # angular velocity of cluster
    omega_c = np.zeros((n, 3))

    # unit vectors in cluster frame 
    for i in range(n): 

        b_data = bound_data[i]
        r_cm = b_data["pos"]
        v_cm = b_data["vel"]

        pos = sim_data[i]["pos"]
        vel = sim_data[i]["vel"]

        obj = cluster_frame(r_cm, v_cm, pos, vel)

        x_hat, y_hat, z_hat, pos_cluster_frame, vel_cluster_frame, omega_vec = obj
        
        c_data={}
        
        c_data["omega"] = omega_vec
        c_data["x_hat"] = x_hat
        c_data["y_hat"] = y_hat
        c_data["z_hat"] = z_hat 
        c_data["pos"]= pos_cluster_frame
        c_data["vel"]=vel_cluster_frame

        cluster_data.append(c_data)
 
    return cluster_data

cluster_data= get_cluster_data(bound_data, sim_data) 

In [ ]:
def get_tidal_data(traj, pot_ext): 

    pos = traj["pos"]
    n=pos.shape[0]

    tidal = np.zeros((n,3))
    max_tidal = np.zeros(n)

    for i in range(n): 

        H = np.zeros((3, 3))
        
        hess = pot_ext.eval(pos[i,:], der=True)
    
    
        # Upper triangular indices
        triu_indices = np.triu_indices(3)
        H[triu_indices] = hess
    
        # Make symmetric
        H = H + H.T - np.diag(np.diag(H))

        tidal[i,:] = np.linalg.eigvals(H)

        max_tidal[i] = np.max(tidal[i,:]) 

    return tidal, max_tidal 

tidal, max_tidal = get_tidal_data(traj, pot_ext)

In [ ]:
# # tidal tensor analysis

# n = traj["pos"].shape[0]
# tidal = np.zeros((n, 3))
# max_tidal=np.zeros(n)
# for i in range(n): 

#     H = np.zeros((3, 3))

#     point = traj[i]["pos"]
    
#     hess = pot_ext.eval(point, der=True)
#     #print(eigen_vals)
#     #print(hess)
    
#     # Upper triangular indices
#     triu_indices = np.triu_indices(3)
#     H[triu_indices] = hess
    
#     # Make symmetric
#     H = H + H.T - np.diag(np.diag(H))

#     tidal[i,:] = np.linalg.eigvals(H)

#     max_tidal[i] = np.max(tidal[i,:]) 
    

In [ ]:
gc_energy = np.zeros(len(bound_data))

i=0
for bdata in bound_data: 

    e = bdata["total energy"]
    gc_energy[i] = e
    i+=1

In [ ]:
#traj["time"]

In [ ]:

max_t=np.max(max_tidal)

fig = plt.figure(figsize=(14, 8 ))
fig.suptitle('Mass Loss, Energy, Max Tidal Force, Radius ', size=20)
ax = fig.add_subplot(411)
ax.set_xlim(tfin.value, 0)
#ax.set_xlim(tfin.value, -900)
ax.set_ylabel(r"$M/M_0$", size=20) 
#ax.set_xlabel("Myr", size=20)

M0=traj["mass"][0]
ax.plot(time_array+tfin.value, traj["mass"]/M0)


ax = fig.add_subplot(412)
ax.set_xlim(tfin.value, 0)
#ax.set_xlim(tfin.value, -900)
ax.set_ylabel(r"$E/|E_0|$", size=20) 
#ax.set_xlabel("Myr", size=20) 
E = traj["energy"]
ax.plot(time_array+tfin.value, E/np.abs(E[0])) 


ax = fig.add_subplot(413)
ax.set_xlim(tfin.value, 0)
#ax.set_xlim(tfin.value, -900)
ax.set_ylabel(r"$\lambda_{max}$ ", size=20) 
ax.set_xlabel("Myr", size=20) 
ax.plot(time_array + tfin.value, max_tidal)


ax = fig.add_subplot(414)
ax.set_xlim(tfin.value, 0)
#ax.set_xlim(tfin.value, -900)
ax.set_ylabel(r"$r(t)$ [kpc]", size=20) 
ax.set_xlabel("Myr", size=20) 
ax.plot(time_array + tfin.value, traj["distance"])

fig.savefig(OUTPUT_DIR / "6569_ml_II.pdf") 

# print("Maybe add distance from galactic center." ) 
# print("How does tidal stripping compare at large tidal tensor versus Particle Spray method.") 
# print("why is there a dip in energy right before an increase?" ) 


In [ ]:
r_traj = np.linalg.norm(traj, axis=1)
r_traj.shape

In [ ]:
# Create figure
fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection='3d')

x = traj[:,0]
y = traj[:,1]
z = traj[:,2]
    
# Create scatter plot with color mapping by radius
ax.plot(x,y,z, alpha = .5) 
ax.set_xlabel('X (pc)')
ax.set_ylabel('Y (pc)')
ax.set_zlabel('Z (pc)')


In [ ]:
sim_data[0]["phi"]

In [ ]:
# def newton(pos,mass, G): 

#     acc=np.zeros_like(pos)

#     phi=np.zeros(pos.shape[0]) 

#     for i in range(pos.shape[0]):
#         for j in range(pos.shape[0]):


#             if i != j: 
            
#                 rij = pos[i] - pos[j] 

#                 phi[i]+= - G*mass[j]/np.linalg.norm(rij)

#                 acc[i]+= - G*mass[j]/np.linalg.norm(rij)**3*rij

#     return acc, phi 
            
# a1, phi1 = newton(pos_0[:10,:], mass[:10], agama.G) 
# a2, phi2 = pyfalcon.gravity(pos_0[:10,:], mass[:10]*agama.G, eps)   

In [ ]:
# # Display animation
# HTML(anim.to_jshtml())

In [ ]:
# Choosing either Plummer or Stone potential
pal5_mass = 2.5e4 * u.Msun
pal5_pot = gp.PlummerPotential(m=pal5_mass, b=4*u.pc, units=galactic)

In [ ]:
# mockstreams
df = ms.ChenStreamDF()

gen_pal5 = ms.MockStreamGenerator(df, pot, progenitor_potential=pal5_pot)

pal5_stream, _ = gen_pal5.run(w0, pal5_mass,dt=-1 * u.Myr, n_steps=1000)

In [ ]:
nt=-1

pos = sim_data[-1]["pos"]
x = pos[:,0]
y = pos[:,1]
z = pos[:,2]

fig = plt.figure(figsize=(14, 4))
plt.suptitle('falcON versus Chen particle Stream', size=30)

ax = fig.add_subplot(131)
ax.scatter(x, y, alpha = .2, label = "falcON")
ax.set_xlabel("x [ kpc]", size=20)
ax.set_ylabel("y [ kpc]", size=20)
ax.scatter(pal5_stream.pos.x,pal5_stream.pos.y, alpha=.05, label = "gala (faint orange)" )

ax.legend(fontsize="large", loc=2)

ax = fig.add_subplot(132)
ax.scatter(x, z, alpha = .2,) 
ax.set_xlabel("x [ kpc]", size=20)
ax.set_ylabel("z [ kpc]", size=20)
ax.scatter(pal5_stream.pos.x,pal5_stream.pos.z, alpha=.075) 

ax = fig.add_subplot(133)
ax.scatter(y, z, alpha = .2,) 
ax.set_xlabel("y [ kpc]", size=20)
ax.set_ylabel("z [ kpc]", size=20)
ax.scatter(pal5_stream.pos.y,pal5_stream.pos.z, alpha=.05) 

plt.tight_layout()
fig.savefig(OUTPUT_DIR / "spray_versus_nbody.pdf") 



In [ ]:
# jacobi radius 

n = traj.shape[0]

r_tidal=[]

for i in range(n): 

    m_sat = bound_data[i]["total mass"]

    x = traj[i,0]
    y = traj[i,1]
    z = traj[i,2]

    vx = v_traj[i,0]
    vy = v_traj[i,1]
    vz = v_traj[i,2]

    # angular momentum components 
    Lx = y * vz - z * vy
    Ly = z * vx - x * vz
    Lz = x * vy - y * vx
    # radius 
    r = np.sqrt(x*x + y*y + z*z)
    # angular momentum magnitude 
    L = np.sqrt(Lx*Lx + Ly*Ly + Lz*Lz)

    # hessian der = pot_host.eval(orbit_sat[:,0:3], der=True)
    der = pot_ext.eval(traj[i,:], der=True)
    d2Phi_dr2 = -(x**2  * der[0] + y**2  * der[1] + z**2  * der[2] +
                  2*x*y * der[3] + 2*y*z * der[4] + 2*z*x * der[5]) / r**2


    # compute the Jacobi radius and the relative velocity at this radius for each point on the trajectory
    Omega = L / r**2
    rj = (agama.G * m_sat / (Omega**2 - d2Phi_dr2))**(1./3)
    #vj = Omega * rj
    #return rj, vj, R

    r_tidal.append(rj)


r_tidal = np.array(r_tidal) 

    
    

In [ ]:
# unbound data 
free_mask = unbound_data[-1]

In [ ]:
# Create figure
fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111)  # Fixed: added subplot number
ax.set_aspect('equal')     # Fixed: proper way to set equal aspect
ax.set_xlabel('X [kpc]', size=20)
ax.set_ylabel('Y [kpc]', size=20)
L = .3
ax.set_xlim(-L, L)
ax.set_ylim(-L, L)

pts, = ax.plot([], [], "o", color="blue", alpha=0.1, markersize=1)  # Added markersize for visibility
free, = ax.plot([], [], "o", color = "red", alpha = .3, markersize=1, label="stripped") 
circ, = ax.plot([], [], color = "orange", alpha =.5 ,label = "Jacobi radius") 
circ2, = ax.plot([], [], color = "black", alpha=.5, label = "energy based bound radius") 
ax.legend(loc=2, fontsize="large") 
plt.close()

theta = np.linspace(0,2*np.pi,1000) 

# Initialization function
def init():
    pts.set_data([], []) 
    circ.set_data([], []) 
    circ2.set_data([], []) 
    free.set_data([], []) 
    return pts, circ, circ2, free,  

def draw(i):  # Fixed: use 'i' instead of undefined 'nt'

    data = sim_data[i]
    pos = data["pos"]

    x = pos[:,0] - np.mean(pos[:,0]) 
    y = pos[:,1] - np.mean(pos[:,1]) 
    #z = pos[:,2] - np.mean(pos[:,2]) 
    
    pts.set_data(x, y)  # Fixed: missing closing bracket and proper syntax

    free_x = x[free_mask]
    free_y = y[free_mask] 

    free.set_data(free_x, free_y) 

    x = r_tidal[i]*np.cos(theta)
    y = r_tidal[i]*np.sin(theta) 

    circ.set_data(x, y)

    r_bound = bound_data[i]["r_bound"]
    
    x = r_bound*np.cos(theta)
    y = r_bound*np.sin(theta)

    circ2.set_data(x, y)

    time = np.round(time_array[i] + tfin.value,0) 
    
    ax.set_title(f'Palomar 5 (X-Y Plane) - Time: {time} Myr')
    
    return pts, circ, circ2,free, 
    
anim = FuncAnimation(fig, draw, init_func=init, frames=len(sim_data), interval=100, blit=True)
#anim.save("pal5test_blob_rj.mp4") 
HTML(anim.to_jshtml())

### Todo

**tidal analysis is likely wrong, need to compute the effective potential and then take the Hessian of that.**

\begin{align}
\vec{a} = - \vec{\nabla} \Phi_g - 2\vec{\Omega} \times \vec{v} - \vec{\Omega} \times (\vec{\Omega} \times \vec{r}) - \frac{d\vec{\Omega}}{dt} \times \vec{r}
\end{align} 

**Energy based boudn radius is not right either. It just give the max radius of the particles... need better method.**

Could I use the viral theorem, find the particles that define a viralized system. Then solve for $R$ by the viral theorem. How do you find the particles that define a virialzed system? '

**Track bound and unbound particles** 

In [ ]:
r_tidal[1]

In [ ]:
bound_data[1]["r_bound"]

In [ ]:
# Lagrange points
n=len(sim_data)

L1 = []
L2 = [] 

def m_encolsed(r, r_cm, pos, mass): 
    ''' enclsoed mass '''

    r_stars = pos - r_cm 
    r_stars = np.linalg.norm(r_stars)

    use = r_stars**2 - r**2 < 0 

    return np.sum(mass[use]) 

def rel_acc(r, r_cm, stars, mass): 
    ''' relative acceleration 
    make sure you get directionc correct'
    acceleration from cluster should point toward r_cm
    acceleration from galaxy should point in opposite direction 
    your search should start on either side of the cluster''
    
    r_mag = np.linalg.norm(r)

    M = m_encolsed(r_mag, r_cm, stars, mass)

    acc = pot_ext.force(r) 

    a_r = np.dot(r_cm, acc)/np.linagl.norm(r_cm) 

    return - agama.G*M/r_mag**2 + a_r

for i in range(n): 

    stars = sim_data[i]["pos"]
    r_cm = traj[i,:]

In [ ]:
traj.shape

In [ ]:
mass.shape

In [ ]:
2**(-13)

In [ ]:
# Create figure
fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111)  # Fixed: added subplot number
ax.set_aspect('equal')     # Fixed: proper way to set equal aspect
ax.set_xlabel('X [kpc]', size=20)
ax.set_ylabel('Y [kpc]', size=20)
L = .3
ax.set_xlim(-L, L)
ax.set_ylim(-L, L)

pts, = ax.plot([], [], "o", color="blue", alpha=0.1, markersize=1)  # Added markersize for visibility
# free, = ax.plot([], [], "o", color = "red", alpha = .3, markersize=1, label="stripped") 
# circ, = ax.plot([], [], color = "orange", alpha =.5 ,label = "Jacobi radius") 
# circ2, = ax.plot([], [], color = "black", alpha=.5, label = "energy based bound radius") 
# ax.legend(loc=2, fontsize="large") 
plt.close()

theta = np.linspace(0,2*np.pi,1000) 

# Initialization function
def init():
    pts.set_data([], []) 
    # circ.set_data([], []) 
    # circ2.set_data([], []) 
    # free.set_data([], []) 
    return pts, #circ, circ2, free,  

def draw(i):  # Fixed: use 'i' instead of undefined 'nt'

    data = cluster_data[i]
    pos = data["pos"]

    x = pos[:,0] - np.mean(pos[:,0]) 
    y = pos[:,1] - np.mean(pos[:,1]) 
    #z = pos[:,2] - np.mean(pos[:,2]) 
    
    pts.set_data(x, y)  # Fixed: missing closing bracket and proper syntax

    # free_x = x[free_mask]
    # free_y = y[free_mask] 

    # free.set_data(free_x, free_y) 

    # x = r_tidal[i]*np.cos(theta)
    # y = r_tidal[i]*np.sin(theta) 

    # circ.set_data(x, y)

    # r_bound = bound_data[i]["r_bound"]
    
    # x = r_bound*np.cos(theta)
    # y = r_bound*np.sin(theta)

    # circ2.set_data(x, y)

    time = np.round(time_array[i] + tfin.value,0) 
    
    ax.set_title(f'Palomar 5 (X-Y Plane) - Time: {time} Myr')
    
    return pts, #circ, circ2,free, 
    
anim = FuncAnimation(fig, draw, init_func=init, frames=len(sim_data), interval=100, blit=True)
#anim.save("pal5test_blob_rj.mp4") 
HTML(anim.to_jshtml())

In [ ]:
len(cluster_data)

In [ ]:
#measure lengths along x, y, and z in cluster frame. 

In [ ]:
mass[0]

In [ ]:
# fix this

def time_scales(V,n): 

    V=V.to(u.km/u.s)
    n=n.to(1/u.pc**3)


    t_s = 4*1e12*(V.value/10)**3/m**2/n.value*u.yr.to(u.Myr)

    print(t_s)

radius=np.linalg.norm(xv[:,:3],axis=1)
rmax =np.max(radius)

vel=np.linalg.norm(xv[:,3:],axis=1)
V = np.std(vel)*u.km/u.s
vol=4/3*np.pi*rmax**3*u.kpc**3
n=radius.size/vol
print(V,n)

time_scales(V,n)

In [ ]:
n.to(1/u.pc**3)

## 1 Gyr Narrative Plots

These cells use the cached `dev3_original_binary_1gyr` products to build presentation-oriented plots. They do not rerun the N-body simulation.


In [ ]:
from pathlib import Path
import runpy

candidate_dirs = [
    Path.cwd(),
    Path.cwd() / "joe's_Code" / "ngc6569",
    Path("/Users/kerrycheon/repos/Work/tidal_stream_research_26/joe's_Code/ngc6569"),
]
NOTEBOOK_DIR = next(path for path in candidate_dirs if (path / "plot_1gyr_narrative.py").exists())
runpy.run_path(str(NOTEBOOK_DIR / "plot_1gyr_narrative.py"), run_name="__main__")


In [ ]:
from IPython.display import Image, display

PLOT_DIR = NOTEBOOK_DIR / "output" / "dev3_original_binary_1gyr" / "narrative_plots"
plot_files = [
    "ngc6569_1gyr_mass_loss_vs_radius.png",
    "ngc6569_1gyr_mass_loss_rate.png",
    "ngc6569_1gyr_sky_colored_by_distance.png",
    "ngc6569_1gyr_sky_colored_by_radial_velocity.png",
    "ngc6569_1gyr_proper_motion_phase_space.png",
    "ngc6569_1gyr_bound_vs_stripped.png",
    "ngc6569_1gyr_initial_vs_final.png",
    "ngc6569_1gyr_parameter_sensitivity.png",
]
for file_name in plot_files:
    path = PLOT_DIR / file_name
    print(path.relative_to(NOTEBOOK_DIR))
    display(Image(filename=str(path), width=900))
